# HiRAG Retrieval q50 Metrics Overview

This notebook explains the retrieval experiment run in `.runs/retrieval_eval/dev_retrieval_q50`.

The benchmark used 50 agriculture queries and compared five context-only retrieval variants:

- `hi`: baseline HiRAG retrieval.
- `hi_weighted`: baseline local retrieval plus query-weighted bridge paths.
- `hi_minmax`: baseline local retrieval plus minmax bridge paths.
- `hi_rerank`: FastEmbed late-interaction local entity reranking plus baseline bridge.
- `hi_rerank_weighted`: reranked local retrieval plus query-weighted bridge paths.

All metrics here are deterministic context/path diagnostics, not answer quality metrics.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

RUN_DIR_CANDIDATES = [
    Path(".runs/retrieval_eval/dev_retrieval_q50"),
    Path("../.runs/retrieval_eval/dev_retrieval_q50"),
]
RUN_DIR = next((path for path in RUN_DIR_CANDIDATES if path.exists()), RUN_DIR_CANDIDATES[0])
metrics_path = RUN_DIR / "metrics.csv"
summary_path = RUN_DIR / "summary.json"

metrics = pd.read_csv(metrics_path)
summary = json.loads(summary_path.read_text())

metrics.shape, metrics["variant"].value_counts().to_dict()

## Strategy Cheat Sheet

The key distinction is where query conditioning happens.

- `hi` selects query-relevant entities, then uses ordinary shortest paths between key entities.
- `hi_weighted` still starts from the same entities, but path edges are weighted by semantic relevance to the query, hub penalty, and edge weight.
- `hi_minmax` uses the same edge costs as weighted retrieval, but chooses paths that minimize the worst edge cost. It is conservative about bad bridge edges.
- `hi_rerank` changes the local entities using ColBERT-style late interaction before the usual bridge.
- `hi_rerank_weighted` combines reranked local entities with weighted bridge paths.

A good variant should increase bridge relevance without exploding context length or latency.

In [ ]:
metric_cols = [
    "latency_seconds",
    "context_tokens",
    "bridge_edge_count",
    "bridge_path_edges",
    "bridge_vs_local_recall",
    "bridge_vs_global_recall",
    "bridge_vs_query_overlap",
    "mean_edge_score",
    "max_edge_cost",
]

summary_table = metrics.groupby("variant")[metric_cols].mean(numeric_only=True)
variant_order = ["hi", "hi_weighted", "hi_minmax", "hi_rerank", "hi_rerank_weighted"]
summary_table = summary_table.reindex(variant_order)
summary_table.round(4)

## Coverage Metrics

These metrics ask whether the bridge text overlaps with the local entity context, the global community context, and the query.

Higher is not automatically better, but if bridge retrieval is working, it should usually overlap more with the query and with both sides of the local/global context.

In [ ]:
coverage_cols = ["bridge_vs_local_recall", "bridge_vs_global_recall", "bridge_vs_query_overlap"]
coverage_long = (
    summary_table[coverage_cols]
    .reset_index()
    .melt(id_vars="variant", var_name="metric", value_name="score")
)

plt.figure(figsize=(12, 5))
sns.barplot(data=coverage_long, x="variant", y="score", hue="metric", order=variant_order)
plt.xticks(rotation=20, ha="right")
plt.title("Bridge Coverage Metrics by Retrieval Strategy")
plt.xlabel("")
plt.ylabel("Mean overlap / recall")
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

Main observation: `hi_minmax` has the strongest deterministic bridge coverage. It improves local recall, global recall, and query overlap. `hi_weighted` and `hi_rerank_weighted` mostly improve query overlap, but not by much. Plain reranking changes the local seed entities, but the coverage signal is mixed.

In [ ]:
cost_cols = ["latency_seconds", "context_tokens", "bridge_edge_count", "bridge_path_edges"]
cost_long = (
    summary_table[cost_cols]
    .reset_index()
    .melt(id_vars="variant", var_name="metric", value_name="value")
)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, metric in zip(axes.flat, cost_cols):
    data = cost_long[cost_long["metric"] == metric]
    sns.barplot(data=data, x="variant", y="value", order=variant_order, ax=ax, color="#4C78A8")
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=25)
plt.suptitle("Cost / Context Size Tradeoffs", y=1.02)
plt.tight_layout()
plt.show()

Cost observation: `hi_minmax` buys better bridge coverage by producing longer paths and larger contexts. This is the central tradeoff. A project-worthy next improvement is budgeted minmax: keep the lower worst-edge cost, but cap path length or bridge tokens.

In [ ]:
base = metrics[metrics.variant == "hi"].set_index("query")
deltas = []
for variant in variant_order[1:]:
    rows = metrics[metrics.variant == variant].set_index("query")
    aligned = rows.join(base, lsuffix="", rsuffix="_hi")
    for col in ["context_tokens", "bridge_edge_count", "bridge_path_edges", "bridge_vs_local_recall", "bridge_vs_global_recall", "bridge_vs_query_overlap", "latency_seconds"]:
        tmp = aligned[[col, f"{col}_hi"]].copy()
        tmp["delta"] = tmp[col] - tmp[f"{col}_hi"]
        tmp["variant"] = variant
        tmp["metric"] = col
        deltas.append(tmp.reset_index()[["query", "variant", "metric", "delta"]])
deltas = pd.concat(deltas, ignore_index=True)

delta_summary = deltas.groupby(["variant", "metric"])["delta"].agg(["mean", "median", lambda s: (s > 0).sum()])
delta_summary = delta_summary.rename(columns={"<lambda_0>": "positive_queries"})
delta_summary.round(4)

In [ ]:
plot_metrics = ["bridge_vs_query_overlap", "bridge_vs_global_recall", "bridge_path_edges", "context_tokens"]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, metric in zip(axes.flat, plot_metrics):
    data = deltas[deltas.metric == metric]
    sns.boxplot(data=data, x="variant", y="delta", order=variant_order[1:], ax=ax)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"Delta vs baseline: {metric}")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

The paired deltas matter more than raw means. `hi_minmax` improves bridge/global recall on most queries, but almost always increases path length and context size. Reranking usually shortens the bridge but also lowers bridge/local recall.

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=metrics,
    x="context_tokens",
    y="bridge_vs_query_overlap",
    hue="variant",
    hue_order=variant_order,
    alpha=0.75,
)
plt.title("Query Overlap vs Context Size")
plt.xlabel("Context tokens")
plt.ylabel("Bridge vs query overlap")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Working Conclusion

- `hi_minmax` is the best evidence that query-conditioned bridge retrieval is useful.
- The current minmax implementation is too permissive about long paths.
- `hi_weighted` is cheaper after edge embeddings are warm, but the quality gain is small.
- Late-interaction reranking is not a clear win by context metrics alone; it needs qualitative/answer-level review.

Next implementation target: budgeted minmax plus persistent edge embedding cache.